# EDA — Santiago Weather Forecast
Análisis exploratorio completo del dataset meteorológico.

**Secciones:**
1. Setup y carga de datos
2. Distribución de precipitación
3. Estacionalidad (mensual y anual)
4. Correlaciones entre variables
5. Variables sinópticas: presión, humedad y nubosidad
6. Outliers y eventos extremos
7. Días secos vs lluviosos
8. Autocorrelación (ACF/PACF)
9. Correlación cruzada con lags
10. Selección del umbral de lluvia (RAIN_THRESHOLD)
11. Resumen de hallazgos


## 1. Setup y carga de datos

In [0]:
%pip install lightgbm --quiet
dbutils.library.restartPython()

In [0]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('/Workspace/Repos/desareca/santiago-weather-forecast')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, pacf
import warnings

warnings.filterwarnings('ignore')

from src.data.ingestion import load_from_delta_table
from src.utils.config import *

# Estilo global
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print('✅ Setup completo')

In [0]:
# Cargar datos crudos
df_raw = load_from_delta_table('weather_raw', spark)

# Renombrar columnas a nombres cortos
rename_map = {
    'precipitation_sum':          'precip',
    'temperature_2m_max':         'temp_max',
    'temperature_2m_min':         'temp_min',
    'temperature_2m_mean':        'temp_mean',
    'windspeed_10m_max':          'viento_max',
    'windgusts_10m_max':          'rafagas_max',
    'winddirection_10m_dominant': 'viento_dir',
    'precipitation_hours':        'precip_horas',
    'shortwave_radiation_sum':    'radiacion',
    'et0_fao_evapotranspiration': 'evapotransp',
    'weathercode':                'weather_code',
}
# Solo renombrar las que existan
rename_map = {k: v for k, v in rename_map.items() if k in df_raw.columns}
df = df_raw.rename(columns=rename_map)

# Ordenar y setear índice
df = df.sort_values('fecha').set_index('fecha')
df.index = pd.to_datetime(df.index)

# Solo usar split de entrenamiento para el EDA (sin data leakage)
n_train = int(len(df) * TRAIN_SPLIT)
df = df.iloc[:n_train].copy()

# Features de calendario
df['anio']    = df.index.year
df['mes']     = df.index.month
df['dia_año'] = df.index.dayofyear
df['estacion'] = df['mes'].map({
    12: 'Verano', 1: 'Verano',  2: 'Verano',
    3:  'Otoño',  4: 'Otoño',   5: 'Otoño',
    6:  'Invierno', 7: 'Invierno', 8: 'Invierno',
    9:  'Primavera', 10: 'Primavera', 11: 'Primavera'
})
df['llueve'] = (df['precip'] > 1.0).astype(int)  # umbral 1mm

print(f'📊 Registros (train): {len(df)}')
print(f'📅 Período: {df.index.min().date()} → {df.index.max().date()}')
print(f'📋 Columnas: {list(df.columns)}')
display(df.describe().round(2))

## 2. Distribución de precipitación

In [0]:
precip = df['precip']
precip_lluvia = precip[precip > 1.0]  # solo días con lluvia real

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Distribución de Precipitación Diaria', fontsize=15, fontweight='bold')

# Panel 1: histograma completo
axes[0].hist(precip, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Todos los días (incluye ceros)')
axes[0].set_xlabel('Precipitación (mm)')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(precip.mean(), color='red', linestyle='--', label=f'Media: {precip.mean():.2f} mm')
axes[0].axvline(precip.median(), color='orange', linestyle='--', label=f'Mediana: {precip.median():.2f} mm')
axes[0].legend()

# Panel 2: solo días con lluvia > 1mm, escala log
axes[1].hist(precip_lluvia, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].set_yscale('log')
axes[1].set_title('Días con lluvia > 1mm (escala log)')
axes[1].set_xlabel('Precipitación (mm)')
axes[1].set_ylabel('Frecuencia (log)')
axes[1].axvline(precip_lluvia.mean(), color='red', linestyle='--', label=f'Media: {precip_lluvia.mean():.1f} mm')
axes[1].legend()

# Panel 3: boxplot por estación
orden_estaciones = ['Verano', 'Otoño', 'Invierno', 'Primavera']
data_box = [df[df['estacion'] == e]['precip'].values for e in orden_estaciones]
bp = axes[2].boxplot(data_box, labels=orden_estaciones, patch_artist=True, showfliers=False)
colores = ['#f4a261', '#e76f51', '#457b9d', '#52b788']
for patch, color in zip(bp['boxes'], colores):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[2].set_title('Distribución por estación')
axes[2].set_ylabel('Precipitación (mm)')

plt.tight_layout()
plt.show()

# Stats clave
print('\n📊 Estadísticas clave:')
print(f'  Total días:          {len(precip)}')
print(f'  Días secos (≤1mm):   {(precip <= 1.0).sum()} ({(precip <= 1.0).mean()*100:.1f}%)')
print(f'  Días con lluvia:     {(precip > 1.0).sum()} ({(precip > 1.0).mean()*100:.1f}%)')
print(f'  Días con >20mm:      {(precip > 20).sum()} ({(precip > 20).mean()*100:.1f}%)')
print(f'  Máximo histórico:    {precip.max():.1f} mm  ({precip.idxmax().date()})')
print(f'  Percentil 95:        {precip.quantile(0.95):.1f} mm')
print(f'  Percentil 99:        {precip.quantile(0.99):.1f} mm')

## 3. Estacionalidad

In [0]:
meses_nombre = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

precip_mensual = df.groupby('mes')['precip'].agg(['mean', 'sum', 'std']).reset_index()
lluvia_mensual = df.groupby('mes')['llueve'].mean().reset_index()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Estacionalidad Mensual de Precipitación — Santiago', fontsize=15, fontweight='bold')

# Panel 1: precipitación media por mes
bars = axes[0, 0].bar(meses_nombre, precip_mensual['mean'],
                       color=['#457b9d' if m in [6,7,8] else '#a8dadc' for m in range(1,13)],
                       edgecolor='white', alpha=0.9)
axes[0, 0].set_title('Precipitación media diaria por mes (mm)')
axes[0, 0].set_ylabel('mm/día')
axes[0, 0].bar_label(bars, fmt='%.1f', padding=2, fontsize=9)

# Panel 2: probabilidad de lluvia por mes
bars2 = axes[0, 1].bar(meses_nombre, lluvia_mensual['llueve'] * 100,
                        color=['#457b9d' if m in [6,7,8] else '#a8dadc' for m in range(1,13)],
                        edgecolor='white', alpha=0.9)
axes[0, 1].set_title('Probabilidad de lluvia por mes (%)')
axes[0, 1].set_ylabel('% días con lluvia > 1mm')
axes[0, 1].bar_label(bars2, fmt='%.1f%%', padding=2, fontsize=9)

# Panel 3: serie temporal anual (media móvil 30 días)
precip_rolling = df['precip'].rolling(30, center=True).mean()
axes[1, 0].fill_between(df.index, 0, df['precip'].values, alpha=0.2, color='steelblue', label='Diario')
axes[1, 0].plot(df.index, precip_rolling, color='#e63946', linewidth=2, label='Media móvil 30d')
axes[1, 0].set_title('Serie temporal con media móvil 30 días')
axes[1, 0].set_ylabel('Precipitación (mm)')
axes[1, 0].legend()

# Panel 4: heatmap mes x año
pivot = df.groupby(['anio', 'mes'])['precip'].sum().unstack()
pivot.columns = meses_nombre
sns.heatmap(pivot, ax=axes[1, 1], cmap='YlGnBu', linewidths=0.5,
            annot=True, fmt='.0f', cbar_kws={'label': 'mm totales'})
axes[1, 1].set_title('Precipitación total mensual por año (mm)')
axes[1, 1].set_xlabel('')

plt.tight_layout()
plt.show()

In [0]:
precip_anual = df.groupby('anio')['precip'].agg(['sum', 'mean', lambda x: (x > 1.0).sum()]).reset_index()
precip_anual.columns = ['anio', 'total_mm', 'media_dia', 'dias_lluvia']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Variabilidad Anual de Precipitación — Santiago', fontsize=15, fontweight='bold')

# Panel 1: total anual
bars = axes[0].bar(precip_anual['anio'].astype(str), precip_anual['total_mm'],
                    color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axhline(precip_anual['total_mm'].mean(), color='red', linestyle='--',
                label=f"Media: {precip_anual['total_mm'].mean():.0f} mm")
axes[0].set_title('Precipitación total anual (mm)')
axes[0].set_ylabel('mm')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()
axes[0].bar_label(bars, fmt='%.0f', padding=2, fontsize=9)

# Panel 2: días con lluvia por año
bars2 = axes[1].bar(precip_anual['anio'].astype(str), precip_anual['dias_lluvia'],
                     color='#52b788', edgecolor='white', alpha=0.85)
axes[1].axhline(precip_anual['dias_lluvia'].mean(), color='red', linestyle='--',
                label=f"Media: {precip_anual['dias_lluvia'].mean():.0f} días")
axes[1].set_title('Días con lluvia por año (>1mm)')
axes[1].set_ylabel('Días')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()
axes[1].bar_label(bars2, fmt='%.0f', padding=2, fontsize=9)

# Panel 3: día del año promedio (estacionalidad media histórica)
precip_dia = df.groupby('dia_año')['precip'].mean().rolling(15, center=True).mean()
axes[2].fill_between(precip_dia.index, 0, precip_dia.values, color='steelblue', alpha=0.7)
for mes_inicio, nombre in zip([1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335],
                                meses_nombre):
    axes[2].axvline(mes_inicio, color='gray', linestyle=':', alpha=0.5, linewidth=0.8)
    axes[2].text(mes_inicio + 5, precip_dia.max() * 0.95, nombre, fontsize=8, color='gray')
axes[2].set_title('Precipitación media por día del año (suavizado 15d)')
axes[2].set_xlabel('Día del año')
axes[2].set_ylabel('mm')

plt.tight_layout()
plt.show()

print('\n📊 Resumen anual:')
display(precip_anual)

## 4. Correlaciones entre variables y precipitación

In [0]:
# Variables numéricas disponibles
vars_modelo = [c for c in df.columns if c not in
               ['anio', 'mes', 'dia_año', 'estacion', 'llueve', 'viento_dir']]

corr_matrix = df[vars_modelo].corr()

fig, ax = plt.subplots(figsize=(20, 20))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, ax=ax,
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    annot=True, fmt='.2f', annot_kws={'size': 9},
    linewidths=0.5, square=True,
    cbar_kws={'shrink': 0.8, 'label': 'Correlación de Pearson'}
)
ax.set_title('Matriz de Correlaciones — Variables Meteorológicas', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# Top correlaciones con precipitación
print('\n🎯 Correlaciones con precipitación (ordenadas):')
corr_precip = corr_matrix['precip'].drop('precip').sort_values(key=abs, ascending=False)
for var, val in corr_precip.items():
    signo = '🔴' if val < -0.3 else ('🟢' if val > 0.3 else '⚪')
    print(f'  {signo} {var:30s}: {val:+.3f}')

In [0]:
# Top 6 variables más correlacionadas con precipitación
top_vars = corr_precip.abs().head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Variables más correlacionadas con precipitación', fontsize=14, fontweight='bold')
axes = axes.flatten()

colores_estacion = {'Verano': '#f4a261', 'Otoño': '#e76f51', 'Invierno': '#457b9d', 'Primavera': '#52b788'}

for i, var in enumerate(top_vars):
    for estacion, color in colores_estacion.items():
        mask = df['estacion'] == estacion
        axes[i].scatter(
            df.loc[mask, var],
            df.loc[mask, 'precip'],
            alpha=0.3, s=10, color=color, label=estacion
        )
    r = corr_matrix.loc[var, 'precip']
    axes[i].set_xlabel(var)
    axes[i].set_ylabel('Precipitación (mm)')
    axes[i].set_title(f'{var}  (r = {r:+.3f})')
    axes[i].set_ylim(-1, df['precip'].quantile(0.995))
    if i == 0:
        axes[i].legend(fontsize=8, markerscale=2)

plt.tight_layout()
plt.show()

## 5. Variables sinópticas: presión, humedad y nubosidad


In [0]:
# Distribución de variables sinópticas: seco vs lluvioso
seco_s     = df[df['llueve'] == 0]
lluvioso_s = df[df['llueve'] == 1]

vars_sinopticas = [c for c in [
    'pressure_msl_mean', 'pressure_trend_24h', 'pressure_range',
    'relative_humidity_2m_mean', 'dew_point_2m_mean', 'vapour_pressure_deficit_mean',
    'cloud_cover_mean', 'cloud_cover_low_mean', 'cloud_cover_mid_mean', 'cloud_cover_high_mean',
    'wind_west_component', 'wind_north_component',
] if c in df.columns]

ncols = 3
nrows = (len(vars_sinopticas) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(20, 5 * nrows))
fig.suptitle('Variables Sinópticas — Distribución Seco vs Lluvioso',
             fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, var in enumerate(vars_sinopticas):
    axes[i].hist(seco_s[var].dropna(), bins=40, alpha=0.6, color='#f4a261',
                 density=True, label=f'Seco (n={len(seco_s)})')
    axes[i].hist(lluvioso_s[var].dropna(), bins=40, alpha=0.6, color='#457b9d',
                 density=True, label=f'Lluvioso (n={len(lluvioso_s)})')
    r = df['precip'].corr(df[var])
    axes[i].set_title(f'{var}  (r={r:+.3f})')
    axes[i].legend(fontsize=8)
    axes[i].set_ylabel('Densidad')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

print('\n📊 Media de variables sinópticas: seco vs lluvioso')
comparativa_s = pd.DataFrame({
    'Seco':       seco_s[vars_sinopticas].mean().round(2),
    'Lluvioso':   lluvioso_s[vars_sinopticas].mean().round(2),
    'Diferencia': (lluvioso_s[vars_sinopticas].mean() - seco_s[vars_sinopticas].mean()).round(2),
})
display(comparativa_s)


In [0]:
# Evolución de variables sinópticas en ventana [-5, +3] días alrededor de eventos de lluvia
eventos = df.index[df['llueve'] == 1]
ventana = list(range(-5, 4))

vars_ventana = [c for c in [
    'pressure_msl_mean', 'pressure_trend_24h',
    'relative_humidity_2m_mean', 'cloud_cover_low_mean', 'dew_point_2m_mean',
] if c in df.columns]

if vars_ventana:
    fig, axes = plt.subplots(1, len(vars_ventana), figsize=(5 * len(vars_ventana), 5))
    if len(vars_ventana) == 1:
        axes = [axes]
    fig.suptitle(
        'Evolución de variables sinópticas alrededor de eventos de lluvia\n(día 0 = día de lluvia)',
        fontsize=13, fontweight='bold'
    )

    n_validos = 0
    for ax, var in zip(axes, vars_ventana):
        series_ventana = []
        for fecha in eventos:
            try:
                vals = [df.loc[fecha + pd.Timedelta(days=d), var] for d in ventana
                        if (fecha + pd.Timedelta(days=d)) in df.index]
                if len(vals) == len(ventana):
                    series_ventana.append(vals)
            except KeyError:
                continue
        if series_ventana:
            arr   = np.array(series_ventana, dtype=float)
            media = arr.mean(axis=0)
            std   = arr.std(axis=0)
            ax.plot(ventana, media, marker='o', color='#457b9d', linewidth=2)
            ax.fill_between(ventana, media - std, media + std, alpha=0.2, color='#457b9d')
            ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Día de lluvia')
            ax.set_title(var, fontsize=10)
            ax.set_xlabel('Días relativos al evento')
            ax.legend(fontsize=8)
            ax.grid(alpha=0.3)
            n_validos = len(series_ventana)

    plt.tight_layout()
    plt.show()
    print(f'  ✅ Eventos analizados: {n_validos}')


## 6. Outliers y eventos extremos


In [0]:
# Umbrales de eventos
p90 = df['precip'].quantile(0.90)
p95 = df['precip'].quantile(0.95)
p99 = df['precip'].quantile(0.99)

extremos = df[df['precip'] >= p95][['precip', 'temp_max', 'temp_min', 'viento_max',
                                     'radiacion', 'evapotransp', 'estacion']].copy()
extremos = extremos.sort_values('precip', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Eventos Extremos de Precipitación', fontsize=14, fontweight='bold')

# Panel 1: serie temporal con eventos marcados
axes[0].fill_between(df.index, 0, df['precip'], alpha=0.3, color='steelblue')
axes[0].plot(df.index, df['precip'], linewidth=0.5, color='steelblue', alpha=0.6)
mask_extremo = df['precip'] >= p95
axes[0].scatter(df.index[mask_extremo], df['precip'][mask_extremo],
                color='red', s=20, zorder=5, label=f'Eventos ≥ p95 ({p95:.1f}mm)')
axes[0].axhline(p90, color='orange', linestyle='--', alpha=0.7, label=f'p90: {p90:.1f}mm')
axes[0].axhline(p95, color='red',    linestyle='--', alpha=0.7, label=f'p95: {p95:.1f}mm')
axes[0].axhline(p99, color='darkred',linestyle='--', alpha=0.7, label=f'p99: {p99:.1f}mm')
axes[0].set_title('Serie temporal con eventos extremos marcados')
axes[0].set_ylabel('Precipitación (mm)')
axes[0].legend(fontsize=9)

# Panel 2: eventos extremos por mes
extremos_mes = df[df['precip'] >= p95].groupby('mes').size()
extremos_mes = extremos_mes.reindex(range(1, 13), fill_value=0)
axes[1].bar(meses_nombre, extremos_mes.values, color='#e63946', edgecolor='white', alpha=0.85)
axes[1].set_title(f'Eventos extremos por mes (≥ p95 = {p95:.1f}mm)')
axes[1].set_ylabel('Cantidad de eventos')

plt.tight_layout()
plt.show()

print(f'\n🌩️  Top 15 eventos extremos (≥ p95 = {p95:.1f}mm):')
display(extremos.head(15))

print(f'\n📊 Resumen por estación:')
display(df[df['precip'] >= p95].groupby('estacion')['precip']
        .agg(['count', 'mean', 'max'])
        .rename(columns={'count': 'n_eventos', 'mean': 'media_mm', 'max': 'max_mm'})
        .round(1))

## 7. Días secos vs lluviosos


In [0]:
seco     = df[df['llueve'] == 0]
lluvioso = df[df['llueve'] == 1]

vars_comparar = [c for c in [
    'temp_max', 'temp_min', 'temp_mean',
    'radiacion', 'evapotransp',
    'viento_max', 'rafagas_max',
    'pressure_msl_mean', 'pressure_trend_24h', 'pressure_range',
    'relative_humidity_2m_mean', 'dew_point_2m_mean', 'vapour_pressure_deficit_mean',
    'cloud_cover_mean', 'cloud_cover_low_mean', 'cloud_cover_high_mean',
    'wind_west_component', 'wind_north_component',
    'precip_horas',
] if c in df.columns]

ncols = 4
nrows = (len(vars_comparar) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(20, 5 * nrows))
fig.suptitle('Distribución de variables: Días Secos vs Lluviosos', fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, var in enumerate(vars_comparar):
    axes[i].hist(seco[var].dropna(), bins=40, alpha=0.6, color='#f4a261',
                 density=True, label=f'Seco (n={len(seco)})')
    axes[i].hist(lluvioso[var].dropna(), bins=40, alpha=0.6, color='#457b9d',
                 density=True, label=f'Lluvioso (n={len(lluvioso)})')
    axes[i].set_title(var)
    axes[i].legend(fontsize=8)
    axes[i].set_ylabel('Densidad')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

print('\n📊 Medias por condición (seco vs lluvioso):')
comparativa = pd.DataFrame({
    'Seco':     seco[vars_comparar].mean(),
    'Lluvioso': lluvioso[vars_comparar].mean(),
}).round(2)
comparativa['Diferencia'] = (comparativa['Lluvioso'] - comparativa['Seco']).round(2)
comparativa['Diferencia_%'] = ((comparativa['Diferencia'] / comparativa['Seco'].abs()) * 100).round(1)
display(comparativa)


In [0]:
# Analizar rachas (streaks)
def calcular_rachas(serie_binaria):
    rachas = []
    contador = 1
    valor_actual = serie_binaria.iloc[0]
    for val in serie_binaria.iloc[1:]:
        if val == valor_actual:
            contador += 1
        else:
            rachas.append({'tipo': 'seco' if valor_actual == 0 else 'lluvia', 'duracion': contador})
            contador = 1
            valor_actual = val
    rachas.append({'tipo': 'seco' if valor_actual == 0 else 'lluvia', 'duracion': contador})
    return pd.DataFrame(rachas)

rachas_df = calcular_rachas(df['llueve'])
rachas_seco   = rachas_df[rachas_df['tipo'] == 'seco']['duracion']
rachas_lluvia = rachas_df[rachas_df['tipo'] == 'lluvia']['duracion']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Rachas de Días Secos y Lluviosos', fontsize=14, fontweight='bold')

axes[0].hist(rachas_seco, bins=40, color='#f4a261', edgecolor='white', alpha=0.85)
axes[0].set_title(f'Rachas secas — media: {rachas_seco.mean():.1f}d, max: {rachas_seco.max()}d')
axes[0].set_xlabel('Duración (días)')
axes[0].set_ylabel('Frecuencia')

axes[1].hist(rachas_lluvia, bins=20, color='#457b9d', edgecolor='white', alpha=0.85)
axes[1].set_title(f'Rachas lluviosas — media: {rachas_lluvia.mean():.1f}d, max: {rachas_lluvia.max()}d')
axes[1].set_xlabel('Duración (días)')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

print(f'\n📊 Estadísticas de rachas:')
print(f'  Rachas secas   — total: {len(rachas_seco)}, media: {rachas_seco.mean():.1f}d, p90: {rachas_seco.quantile(0.9):.0f}d, max: {rachas_seco.max()}d')
print(f'  Rachas lluvia  — total: {len(rachas_lluvia)}, media: {rachas_lluvia.mean():.1f}d, p90: {rachas_lluvia.quantile(0.9):.0f}d, max: {rachas_lluvia.max()}d')

## 8. Autocorrelación (ACF/PACF)


In [0]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle('Autocorrelación de Precipitación Diaria', fontsize=14, fontweight='bold')

# ACF / PACF — todos los días
plot_acf(df['precip'].fillna(0),  lags=60, ax=axes[0, 0], title='ACF — Todos los días (60 lags)', alpha=0.05)
plot_pacf(df['precip'].fillna(0), lags=60, ax=axes[0, 1], title='PACF — Todos los días (60 lags)', alpha=0.05, method='ywm')

# ACF / PACF — solo días lluviosos (serie binaria)
plot_acf(df['llueve'],  lags=60, ax=axes[1, 0], title='ACF — Indicador lluvia binario (60 lags)', alpha=0.05)
plot_pacf(df['llueve'], lags=60, ax=axes[1, 1], title='PACF — Indicador lluvia binario (60 lags)', alpha=0.05, method='ywm')

plt.tight_layout()
plt.show()

## 9. Correlación cruzada con lags


In [0]:
# Correlación cruzada: var(t-lag) → precip(t), para lag 1..14
vars_lag = [c for c in [
    'temp_max', 'temp_min', 'temp_mean',
    'viento_max', 'rafagas_max',
    'radiacion', 'evapotransp', 'weather_code',
    'pressure_msl_mean', 'pressure_trend_24h', 'pressure_range',
    'relative_humidity_2m_mean', 'dew_point_2m_mean', 'vapour_pressure_deficit_mean',
    'cloud_cover_mean', 'cloud_cover_low_mean', 'cloud_cover_high_mean',
    'wind_west_component', 'wind_north_component',
] if c in df.columns]

lags_range = range(1, 15)
corr_lag_df = pd.DataFrame(index=lags_range, columns=vars_lag, dtype=float)

for lag in lags_range:
    for var in vars_lag:
        corr_lag_df.loc[lag, var] = df['precip'].corr(df[var].shift(lag))

corr_lag_df['precip_lag'] = [df['precip'].corr(df['precip'].shift(lag)) for lag in lags_range]

grupos = {
    'Nubosidad + Humedad': [c for c in ['cloud_cover_mean', 'cloud_cover_low_mean',
                                         'cloud_cover_high_mean', 'relative_humidity_2m_mean',
                                         'dew_point_2m_mean', 'vapour_pressure_deficit_mean']
                             if c in corr_lag_df.columns],
    'Presión': [c for c in ['pressure_msl_mean', 'pressure_trend_24h', 'pressure_range']
                if c in corr_lag_df.columns],
    'Temperatura + Radiación': [c for c in ['temp_max', 'temp_min', 'radiacion', 'evapotransp']
                                 if c in corr_lag_df.columns],
    'Viento + Otros': [c for c in ['viento_max', 'rafagas_max', 'wind_west_component',
                                    'wind_north_component', 'weather_code', 'precip_lag']
                       if c in corr_lag_df.columns],
}

fig, axes = plt.subplots(2, 2, figsize=(20, 12))
fig.suptitle('Correlación cruzada: var(t-lag) → precip(t)', fontsize=14, fontweight='bold')
axes = axes.flatten()

for idx_g, (titulo, cols) in enumerate(grupos.items()):
    ax = axes[idx_g]
    for col in cols:
        ls = '--' if col == 'precip_lag' else '-'
        ax.plot(corr_lag_df.index, corr_lag_df[col], marker='o', markersize=4,
                label=col, linestyle=ls, linewidth=1.8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axhline(0.1, color='gray', linestyle=':', alpha=0.5)
    ax.axhline(-0.1, color='gray', linestyle=':', alpha=0.5)
    ax.set_title(titulo)
    ax.set_xlabel('Lag (días)')
    ax.set_ylabel('Correlación con precip(t)')
    ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
    ax.set_xticks(list(lags_range))
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('\n🎯 Correlación en lag=1 (día anterior → precipitación hoy):')
lag1 = corr_lag_df.loc[1].sort_values(key=abs, ascending=False)
for var, val in lag1.items():
    signo = '🔴' if val < -0.15 else ('🟢' if val > 0.15 else '⚪')
    print(f'  {signo} {var:35s}: {val:+.3f}')


## 10. Selección del umbral de lluvia (RAIN_THRESHOLD)

Tres análisis para determinar el umbral óptimo que separa "seco" de "lluvia":

1. **Distribución frontera** — cuántos días caen en cada rango de precipitación
2. **Señal predictiva por umbral** — qué tan predecible es lluvia mañana dado que llovió hoy (autocorrelación condicional)
3. **Estabilidad del clasificador** — F1 en CV para distintos umbrales con una config fija


In [0]:
# ── 1. Distribución frontera ──────────────────────────────────────
CANDIDATE_THRESHOLDS = [0.1, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0]
precip = df['precip']
n_total = len(precip)

print('='*60)
print('DISTRIBUCIÓN POR RANGO DE PRECIPITACIÓN')
print('='*60)
print(f'  {"Rango":20s}  {"Días":>6}  {"% total":>8}  {"Acum. seco":>12}')
print(f'  {"-"*52}')

rangos = [
    ('= 0.0 mm (sin lluvia)',  (precip == 0)),
    ('0.1 – 0.9 mm (traza)',   (precip > 0)   & (precip < 1.0)),
    ('1.0 – 1.9 mm',           (precip >= 1.0) & (precip < 2.0)),
    ('2.0 – 2.9 mm',           (precip >= 2.0) & (precip < 3.0)),
    ('3.0 – 4.9 mm',           (precip >= 3.0) & (precip < 5.0)),
    ('5.0 – 9.9 mm',           (precip >= 5.0) & (precip < 10.0)),
    ('10 – 19.9 mm',           (precip >= 10)  & (precip < 20.0)),
    ('≥ 20 mm (pico)',         (precip >= 20)),
]

acum = 0
for label, mask in rangos:
    n = mask.sum()
    pct = n / n_total * 100
    acum += pct
    print(f'  {label:20s}  {n:>6d}  {pct:>7.1f}%  {acum:>11.1f}%')

print(f'\n  Total: {n_total} días')

# Resumen por umbral candidato
print(f'\n  Días "secos" según cada umbral candidato:')
print(f'  {"Umbral":>10}  {"Días secos":>10}  {"% secos":>9}  {"Días lluvia":>12}  {"% lluvia":>9}')
print(f'  {"-"*56}')
for thr in CANDIDATE_THRESHOLDS:
    n_seco   = (precip <= thr).sum()
    n_lluvia = (precip >  thr).sum()
    print(f'  {thr:>9.1f}  {n_seco:>10d}  {n_seco/n_total*100:>8.1f}%'
          f'  {n_lluvia:>12d}  {n_lluvia/n_total*100:>8.1f}%')

In [0]:
# ── 2. Señal predictiva por umbral ────────────────────────────────
# Para cada umbral: P(llueve hoy | llovió ayer > umbral) vs P(llueve hoy | no llovió ayer)
# Una señal fuerte = la diferencia entre ambas probabilidades es grande

print('='*60)
print('SEÑAL PREDICTIVA POR UMBRAL (autocorrelación condicional)')
print('='*60)
print(f'  {"Umbral":>8}  {"P(lluvia|llovió ayer)":>22}  {"P(lluvia|seco ayer)":>21}  {"Lift":>8}  {"Señal":>6}')
print(f'  {"-"*72}')

precip_hoy   = precip.values[1:]
precip_ayer  = precip.values[:-1]
lifts = {}

for thr in CANDIDATE_THRESHOLDS:
    llovio_ayer = precip_ayer > thr
    llueve_hoy  = precip_hoy  > thr

    p_dado_lluvia = llueve_hoy[llovio_ayer].mean() if llovio_ayer.sum() > 0 else np.nan
    p_dado_seco   = llueve_hoy[~llovio_ayer].mean() if (~llovio_ayer).sum() > 0 else np.nan
    lift          = p_dado_lluvia / p_dado_seco if p_dado_seco > 0 else np.nan
    lifts[thr]    = lift

    barra = '█' * int(lift * 3) if not np.isnan(lift) else ''
    print(f'  {thr:>7.1f}  {p_dado_lluvia:>21.1%}  {p_dado_seco:>21.1%}'
          f'  {lift:>7.2f}x  {barra}')

mejor_lift = max(lifts, key=lambda t: lifts[t] if not np.isnan(lifts[t]) else -1)
print(f'\n  → Umbral con mayor señal predictiva (lift máximo): {mejor_lift} mm')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Señal predictiva por umbral', fontsize=13, fontweight='bold')

thrs   = CANDIDATE_THRESHOLDS
p_si   = []
p_no   = []
lifts_ = []

for thr in thrs:
    llovio = precip_ayer > thr
    hoy    = precip_hoy  > thr
    p_si.append(hoy[llovio].mean()  if llovio.sum()  > 0 else np.nan)
    p_no.append(hoy[~llovio].mean() if (~llovio).sum() > 0 else np.nan)
    lifts_.append(lifts[thr])

x = range(len(thrs))
axes[0].plot(x, p_si, marker='o', label='P(lluvia | llovió ayer)', color='steelblue', linewidth=2)
axes[0].plot(x, p_no, marker='s', label='P(lluvia | seco ayer)',   color='#e63946',   linewidth=2)
axes[0].fill_between(x, p_no, p_si, alpha=0.15, color='steelblue')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'{t}mm' for t in thrs])
axes[0].set_ylabel('Probabilidad')
axes[0].set_title('Probabilidades condicionales')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].bar(x, lifts_, color=['#e63946' if t == mejor_lift else 'steelblue' for t in thrs],
            edgecolor='white', alpha=0.85)
for i, v in enumerate(lifts_):
    axes[1].text(i, v + 0.05, f'{v:.2f}x', ha='center', fontsize=9)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'{t}mm' for t in thrs])
axes[1].set_ylabel('Lift')
axes[1].set_title('Lift predictivo (P_si / P_no)')
axes[1].grid(alpha=0.3, axis='y')
axes[1].axhline(1.0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [0]:
# Construir df_train con el pipeline completo
# El df del EDA es crudo (sin features ni target del modelo)
from src.data.preprocessing import build_features, train_test_split as tts
from src.evaluation.two_stage_cv import run_cv_clf_grid
from src.utils.config import N_SPLITS, TEST_SIZE, MIN_TRAIN_SIZE, CLF_THRESHOLDS

df_pipeline          = build_features(df_raw)
df_train_pipeline, _ = tts(df_pipeline)

print(f'✅ df_train_pipeline: {df_train_pipeline.shape}')
print(f'   Target: {[c for c in df_train_pipeline.columns if "precip_t1" in c]}')
print(f'   Período: {df_train_pipeline.index.min().date()} → {df_train_pipeline.index.max().date()}')

In [0]:
# ── 3. Estabilidad del clasificador por umbral ────────────────────
# Corre run_cv_clf_grid con una config fija bajo distintos clf_rain_threshold
# Compara avg_Fbeta y std_Fbeta para encontrar el umbral mm más estable

# Config fija — punto de partida neutro
CLF_FIXED = [{
    'name': 'clf_fixed',
    'n_estimators':      200,
    'learning_rate':     0.05,
    'max_depth':         6,
    'num_leaves':        31,
    'min_child_samples': 20,
    'subsample':         0.8,
    'colsample_bytree':  0.8,
}]

# Umbrales en mm a evaluar como clf_rain_threshold
CLF_MM_TO_TEST = [0.1, 0.2, 0.3, 0.4, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]
threshold_results = []

for clf_mm in CLF_MM_TO_TEST:
    print(f'\n>>> Probando CLF_RAIN_THRESHOLD = {clf_mm} mm...')
    res = run_cv_clf_grid(
        clf_grid=CLF_FIXED,
        df=df_train_pipeline,
        thresholds=CLF_THRESHOLDS,          # lista de umbrales de prob: [0.2, 0.3, 0.4, 0.5]
        clf_rain_threshold=clf_mm,          # umbral mm para etiquetar positivos del clf
        n_splits=N_SPLITS,
        test_size=TEST_SIZE,
        min_train_size=MIN_TRAIN_SIZE,
        log_mlflow=False,
    )
    row = res.iloc[0]
    threshold_results.append({
        'clf_rain_threshold':  clf_mm,
        'avg_f1':              row['avg_f1'],
        'std_f1':              row['std_f1'],
        'avg_fbeta':           row['avg_fbeta'],
        'std_fbeta':           row['std_fbeta'],
        'avg_auc':             row['avg_auc'],
        'avg_precision':       row['avg_precision'],
        'avg_recall':          row['avg_recall'],
        'best_clf_threshold':  row['best_threshold'],
    })

df_thr = pd.DataFrame(threshold_results)

print('\n' + '='*65)
print('COMPARATIVA DE CLF_RAIN_THRESHOLD — Fbeta en CV')
print('='*65)
print(df_thr[['clf_rain_threshold', 'avg_fbeta', 'std_fbeta',
              'avg_auc', 'avg_precision', 'avg_recall',
              'best_clf_threshold']].to_string(index=False))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Impacto de CLF_RAIN_THRESHOLD en el clasificador', fontsize=13, fontweight='bold')

x = range(len(CLF_MM_TO_TEST))

axes[0].errorbar(
    x, df_thr['avg_fbeta'], yerr=df_thr['std_fbeta'],
    marker='o', linewidth=2, capsize=5,
    color='steelblue', label='avg_Fbeta ± std'
)
axes[0].plot(x, df_thr['avg_precision'], marker='s', linewidth=1.5,
             color='#52b788', label='avg_Precision', linestyle='--')
axes[0].plot(x, df_thr['avg_recall'],    marker='^', linewidth=1.5,
             color='#e63946', label='avg_Recall', linestyle='--')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'{t}mm' for t in CLF_MM_TO_TEST])
axes[0].set_ylabel('Métrica')
axes[0].set_title('Fbeta, Precision, Recall por umbral mm')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

axes[1].bar(x, df_thr['std_fbeta'],
            color=['#e63946' if v == df_thr['std_fbeta'].min() else 'steelblue'
                   for v in df_thr['std_fbeta']],
            edgecolor='white', alpha=0.85)
for i, v in enumerate(df_thr['std_fbeta']):
    axes[1].text(i, v + 0.002, f'{v:.3f}', ha='center', fontsize=9)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'{t}mm' for t in CLF_MM_TO_TEST])
axes[1].set_ylabel('std Fbeta (entre folds)')
axes[1].set_title('Estabilidad por umbral mm (menor std = más estable)')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [0]:
# ── Recomendación automática ──────────────────────────────────────
print('='*60)
print('RECOMENDACIÓN DE RAIN_THRESHOLD')
print('='*60)

# Score combinado: premia F1 alto y std bajo
# score = avg_f1 - std_f1  (mayor es mejor)
df_thr['score'] = df_thr['avg_fbeta'] - df_thr['std_fbeta']
best_row = df_thr.loc[df_thr['score'].idxmax()]

print(f'\n  Umbral recomendado: {best_row["clf_rain_threshold"]} mm')
print(f'  avg_fbeta: {best_row["avg_fbeta"]:.3f}')
print(f'  std_fbeta: {best_row["std_fbeta"]:.3f}  (estabilidad entre folds)')
print(f'  avg_AUC:   {best_row["avg_auc"]:.3f}')
print(f'  Precision: {best_row["avg_precision"]:.3f}')
print(f'  Recall:    {best_row["avg_recall"]:.3f}')

print(f'\n  Señal predictiva (lift) más alta en: {mejor_lift} mm')

print(f'\n  Tabla completa (ordenada por score):')
print(df_thr[['clf_rain_threshold', 'avg_fbeta', 'std_fbeta', 'score',
              'avg_precision', 'avg_recall']].sort_values('score', ascending=False).to_string(index=False))

print(f'\n{"="*60}')
print(f'  → Actualizar config.py:  RAIN_THRESHOLD = {best_row["clf_rain_threshold"]}')
print(f'{"="*60}')

## 11. Resumen de hallazgos


In [0]:
print('=' * 65)
print('RESUMEN DE HALLAZGOS EDA')
print('=' * 65)

pct_seco    = (df['precip'] <= 1.0).mean() * 100
pct_lluvia  = (df['precip'] > 1.0).mean() * 100
pct_extremo = (df['precip'] >= p95).mean() * 100

print(f'''
1. DISTRIBUCIÓN
   • {pct_seco:.1f}% días secos (≤1mm) — desbalanceo severo
   • {pct_lluvia:.1f}% días con lluvia real (>1mm)
   • {pct_extremo:.1f}% días con eventos extremos (≥p95={p95:.1f}mm)
   • Distribución muy asimétrica (cola derecha larga)

2. ESTACIONALIDAD
   • Patrón mediterráneo claro: lluvia concentrada en invierno (Jun-Ago)
   • Verano (Dic-Feb) prácticamente seco
   • Variabilidad interanual alta — años La Niña (2020-2023) muy secos

3. CORRELACIONES EN EL MISMO DÍA (t)
   • precip_horas (+0.76): leakage intradía — NO usar como feature
   • cloud_cover_mid_mean (+0.51): nubosidad media
   • weather_code (+0.50): tipo de evento WMO
   • cloud_cover_low_mean (+0.36): nubosidad baja (lluvia frontal)
   • relative_humidity_2m_mean (+0.32): humedad relativa

4. CORRELACIONES CON LAG (t-1 → t)
   • weather_code (lag=1): señal más fuerte entre variables diarias
   • radiacion / evapotransp: señal inversa persistente hasta lag~3
   • cloud_cover_low_mean: señal positiva en lag=1-2
   • pressure_trend_24h: caída de presión el día anterior precede lluvia
   • dew_point_2m_mean: temperatura de rocío alta precede lluvia

5. VARIABLES SINÓPTICAS
   • pressure_trend_24h: diferencia clara seco vs lluvioso
   • cloud_cover_low_mean: separación marcada entre distribuciones
   • relative_humidity_2m_mean: ~65% seco vs ~85% lluvioso
   • wind_west_component: viento del oeste más frecuente en lluvia
   • Evolución: presión cae 2-3 días antes del evento de lluvia

6. RACHAS
   • Rachas secas: media 12.7d, p90=27d, max=138d
   • Rachas lluviosas: media 1.6d, rara vez superan 3 días consecutivos

7. IMPLICACIONES PARA EL MODELO
   • Features prioritarias lag=1:
     weather_code, cloud_cover_low_mean, radiacion, evapotransp,
     pressure_trend_24h, relative_humidity_2m_mean, dew_point_2m_mean
   • Rolling windows de presión y humedad capturan régimen sinóptico
   • Estacionalidad cíclica (mes_sin/cos) — patrón muy marcado
   • NO usar precip_horas (leakage intradía)
   • Agregar días_secos_consecutivos como feature de racha
''')
print('=' * 65)
